```markdown
# Análisis y Limpieza de Datos de Spotify

Este cuaderno de Jupyter documenta el proceso de análisis y limpieza de un conjunto de datos de canciones de Spotify. A continuación se describen los pasos realizados y el propósito de cada uno.

## 1. Cargar Datos Iniciales

```python
import pandas as pd

file_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\final_url6.csv"
df_clean = pd.read_csv(file_path, low_memory=False)
```

Se carga el archivo CSV `final_url6.csv` en un DataFrame de pandas para su posterior análisis.

## 2. Selección y Análisis de Columnas de Interés

```python
# Crear un nuevo dataframe con las columnas de interés
columns_to_check = ['album_name', 'popularity', 'album_release_date', 'spotify_url', 'duration_ms']
df_no_url = df_clean[columns_to_check]

# Contar nulos en cada columna
null_counts = df_no_url.isnull().sum()

# Mostrar resultados
print("Número de nulos por columna:")
print(null_counts)

# Verificar si todas las columnas tienen el mismo número de nulos
same_nulls = null_counts.nunique() == 1
print(f"\n¿Tienen todas las columnas el mismo número de nulos? {'Sí' if same_nulls else 'No'}")
print(f"\nParece igualmente que es caso de urls no encontradas, voy a pasarlas de nuevo a buscar urls por si hubiera suerte")
```

Se seleccionan las columnas de interés y se cuenta el número de valores nulos en cada una de ellas. Además, se verifica si todas las columnas tienen el mismo número de nulos.

## 3. Filtrado de Filas con Valores Nulos

```python
# Crear un dataframe con filas que tienen valores nulos en 'spotify_url'
df_no_url = df_clean[df_clean[['spotify_url', 'duration_ms']].isnull().any(axis=1)]

# Verificar las primeras filas del nuevo dataframe
print(df_no_url.head())

# Guardar el nuevo dataframe como un archivo CSV
df_no_url.to_csv('df_no_url.csv', index=False)
print("El archivo `df_no_url.csv` ha sido guardado exitosamente.")
```

Se crea un nuevo DataFrame que contiene solo las filas con valores nulos en las columnas `spotify_url` o `duration_ms`. Este DataFrame se guarda en un archivo CSV para su posterior análisis.

## 4. Actualización de Datos con Nuevas URLs

```python
import pandas as pd

# Cargar los archivos
final_idiomas_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\final_url5.csv"
df_new_url_path = r"C:\Users\solan\Downloads\get_data_from_songs\src\df_url5.csv"
output_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\final_url6.csv"

final_idiomas = pd.read_csv(final_idiomas_path)
df_new_url = pd.read_csv(df_new_url_path)

# Columnas que queremos actualizar
columns_to_update = ['album_name', 'popularity', 'album_release_date', 'spotify_url', 'duration_ms']

# Fusionar los datasets por `recording_id`
df_merged = final_idiomas.merge(df_new_url[['recording_id'] + columns_to_update], 
                                on='recording_id', how='left', suffixes=('', '_new'))

# Reemplazar solo los valores nulos en `final_idiomas`
for col in columns_to_update:
    df_merged[col] = df_merged[col].combine_first(df_merged[col + '_new'])
    df_merged.drop(columns=[col + '_new'], inplace=True)  # Eliminar columnas auxiliares

# Guardar el DataFrame actualizado
df_merged.to_csv(output_path, index=False)

print(f"✅ Archivo actualizado guardado en: {output_path}")
```

Se cargan dos archivos CSV y se fusionan para actualizar las columnas de interés con nuevas URLs. Los valores nulos en el DataFrame original se reemplazan con los valores correspondientes del nuevo DataFrame.

## 5. Verificación de Datos Actualizados

```python
df = pd.read_csv(output_path, low_memory=False)
```

Se carga el archivo CSV actualizado para verificar los cambios realizados.

## 6. Análisis de Valores Nulos

```python
print("Valores nulos por columna:")
print(df.isnull().sum())
```

Se cuentan y muestran los valores nulos por columna en el DataFrame actualizado.

```python
null_values = df.isnull().sum()
null_values = null_values[null_values > 0]  # Filtrar solo las columnas con nulos
print(null_values.sort_values(ascending=False))  # Ordenar de mayor a menor
```

Se filtran y ordenan las columnas que aún contienen valores nulos.

## 7. Análisis de Filas Duplicadas

```python
print("\nNúmero total de filas duplicadas:", df.duplicated().sum())
```

Se cuenta y muestra el número total de filas duplicadas en el DataFrame.

Este cuaderno proporciona un flujo de trabajo completo para la limpieza y actualización de datos de canciones de Spotify, asegurando que los datos estén lo más completos y precisos posible.
```

In [1]:
import pandas as pd

file_path = r"C:\Users\solan\MoodTune\data\procesando\df_80-200_p6_last.csv"
df = pd.read_csv(file_path, low_memory=False)

In [12]:
# Crear un nuevo dataframe con las columnas de interés
columns_to_check = ['album_name', 'popularity', 'album_release_date', 'spotify_url', 'duration_ms']
df_no_url = df[columns_to_check]

# Contar nulos en cada columna
null_counts = df_no_url.isnull().sum()

# Mostrar resultados
print("Número de nulos por columna:")
print(null_counts)

# Verificar si todas las columnas tienen el mismo número de nulos
same_nulls = null_counts.nunique() == 1
print(f"\n¿Tienen todas las columnas el mismo número de nulos? {'Sí' if same_nulls else 'No'}")
print(f"\nParece igualmente que es caso de urls no encontradas, voy a pasarlas de nuevo a buscar urls por si hubiera suerte")


Número de nulos por columna:
album_name            5090
popularity            5086
album_release_date    5086
spotify_url           5086
duration_ms           5086
dtype: int64

¿Tienen todas las columnas el mismo número de nulos? No

Parece igualmente que es caso de urls no encontradas, voy a pasarlas de nuevo a buscar urls por si hubiera suerte


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26923 entries, 0 to 26922
Data columns (total 84 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   song_name                           26923 non-null  object 
 1   artist_name                         26923 non-null  object 
 2   recording_id                        26923 non-null  object 
 3   danceable                           26923 non-null  float64
 4   not_danceable                       26923 non-null  float64
 5   male                                26923 non-null  float64
 6   female                              26923 non-null  float64
 7   timbre_bright                       26923 non-null  float64
 8   timbre_dark                         26923 non-null  float64
 9   tonal                               26923 non-null  float64
 10  atonal                              26923 non-null  float64
 11  instrumental                        26923

In [5]:
# crear df solo con las que no tengan nulo en spotify_url
df_no_url3 = df[df['spotify_url'].notnull()]
df_no_url3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21981 entries, 0 to 25001
Data columns (total 86 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   song_name                           21981 non-null  object 
 1   artist_name                         21981 non-null  object 
 2   recording_id                        21981 non-null  object 
 3   danceable                           21981 non-null  float64
 4   not_danceable                       21981 non-null  float64
 5   male                                21981 non-null  float64
 6   female                              21981 non-null  float64
 7   timbre_bright                       21981 non-null  float64
 8   timbre_dark                         21981 non-null  float64
 9   tonal                               21981 non-null  float64
 10  atonal                              21981 non-null  float64
 11  instrumental                        21981 non-

In [24]:
# ver todo el output como scroll
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [25]:
# QUITAR DEL DATASET LAS QUE TENGAN spotify_url nulo
df_no_url = df_no_url.dropna(subset=['spotify_url'])

In [27]:
# Contar nulos en cada columna
null_counts = df_no_url.isnull().sum()

In [29]:
df_no_url.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12122 entries, 0 to 12655
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   album_name          12122 non-null  object 
 1   popularity          12122 non-null  float64
 2   album_release_date  12122 non-null  object 
 3   spotify_url         12122 non-null  object 
 4   duration_ms         12122 non-null  float64
dtypes: float64(2), object(3)
memory usage: 568.2+ KB


In [36]:
# guardar como 0_to_translate 
df_no_url.to_csv(r"C:\Users\solan\MoodTune\data\procesando\0_to_translate.csv", index=False)

In [26]:

df_clean['language'].value_counts()

language
en     11037
es       467
de       291
fr       227
pt       120
sv       101
it        42
ja        42
pl        31
nl        21
tr        16
no        14
da        11
ko        10
ca         9
fi         9
so         7
fil        7
hi         7
la         4
ig         3
ms         3
ru         3
haw        2
ro         2
mt         2
af         2
ht         2
ne         2
fa         2
zh         2
id         2
et         2
eu         2
sl         2
ga         2
hu         2
bn         1
hr         1
jv         1
az         1
eo         1
mg         1
is         1
ha         1
zu         1
yo         1
co         1
gl         1
sm         1
mi         1
th         1
Name: count, dtype: int64

In [4]:
df_clean.head()

,artist_name,song_name,recording_id,danceable,not_danceable,male,female,timbre_bright,timbre_dark,tonal,...,lyrics,spotify_url,album_name,album_release_date,duration_ms,popularity,language,genres,tags,views
0,morning teleportation,banjo disco,f8480eff-db6b-4272-98ef-281e3ed5ab95,0.883,0.117,0.015,0.985,0.964,0.036,0.974,...,cant remember how we got here now but its so r...,https://open.spotify.com/track/65E9dg0Cy2iGtQo...,Expanding Anyway,2014-03-03,265960.0,6.0,en,NaN,NaN,NaN
1,don omar,los hombres tienen la culpa,d8dee02d-ef6e-4d2f-91db-6b34c61eb69d,0.928,0.072,0.000,1.000,0.986,0.014,0.391,...,si mi culpabilidad es por una razón justa si e...,https://open.spotify.com/track/5HTycUV2xsjv4Mf...,Los Cocorocos,2006-01-01,285373.0,28.0,es,NaN,NaN,NaN
2,blowsight,invisible ink,20c60ccf-ab87-4296-8d56-ce83084fa2ea,0.961,0.039,0.044,0.956,0.006,0.994,0.034,...,did you know that i knew all the things that y...,https://open.spotify.com/track/4ByqoEqSZHOysSq...,Dystopia Lane,2011-07-01,244924.0,19.0,en,NaN,NaN,NaN
3,the red jumpsuit apparatus,wide is the gate,0a60eac7-0c81-461c-a9e6-64a006e08253,0.759,0.241,0.079,0.921,0.112,0.888,0.984,...,wide is the gate lyrics were not the weak or t...,https://open.spotify.com/track/0pFC6xwt610Drnl...,"Et Tu, Brute ?",2013-03-15,178606.0,6.0,en,NaN,NaN,NaN
4,chasing safety,brand new prison,a5cff947-46c3-4614-aeb7-84038293419c,0.394,0.606,0.240,0.760,0.027,0.973,0.232,...,this is a lonely road im not sure where it goe...,https://open.spotify.com/track/6zczwaNzuqe5nLs...,Nomad,2017-01-06,240540.0,8.0,en,NaN,NaN,NaN


In [8]:
# Crear un dataframe con filas que tienen valores nulos en 'spotify_url'
df_no_url = df[df[['spotify_url', 'duration_ms']].isnull().any(axis=1)]

# Verificar las primeras filas del nuevo dataframe
print(df_no_url.head())

# Guardar el nuevo dataframe como un archivo CSV
df_no_url.to_csv('df_no_url.csv', index=False)
print("El archivo `df_no_url.csv` ha sido guardado exitosamente.")


                            song_name    artist_name  \
3                         Blue Monday   Lisa Germano   
6               Born Under a Bad Sign            MDC   
8   Story of a Life alternate version   Harry Chapin   
11                            Machine      Nomeansno   
19                          Evilution  Oliver Magnum   

                            recording_id  danceable  not_danceable   male  \
3   1fef8bdd-285a-4948-be33-d994f7d28325      0.000          1.000  0.622   
6   fe9323b3-32c5-4706-8f14-a5099937b725      0.000          1.000  0.622   
8   ca9521bd-03b9-4e73-8e88-8fc819dee2da      0.442          0.558  0.431   
11  ceee9927-1d56-4ae8-853a-463381290427      1.000          0.000  0.595   
19  18684efb-8068-4cbc-bf62-e4ff8aa8c71e      0.849          0.151  0.131   

    female  timbre_bright  timbre_dark  tonal  ...  album_release_date  \
3    0.378          0.004        0.996  0.000  ...                 NaN   
6    0.378          0.062        0.938  0.031  ...  

pasarlas por spotify spoty nuevas

In [42]:
import pandas as pd

# Cargar los archivos
final_idiomas_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\final_url5.csv"
df_new_url_path = r"C:\Users\solan\Downloads\get_data_from_songs\src\df_url5.csv"
output_path = r"C:\Users\solan\Downloads\get_data_from_songs\data\final_url6.csv"

final_idiomas = pd.read_csv(final_idiomas_path)
df_new_url = pd.read_csv(df_new_url_path)

# Columnas que queremos actualizar
columns_to_update = ['album_name', 'popularity', 'album_release_date', 'spotify_url', 'duration_ms']

# Fusionar los datasets por `recording_id`
df_merged = final_idiomas.merge(df_new_url[['recording_id'] + columns_to_update], 
                                on='recording_id', how='left', suffixes=('', '_new'))

# Reemplazar solo los valores nulos en `final_idiomas`
for col in columns_to_update:
    df_merged[col] = df_merged[col].combine_first(df_merged[col + '_new'])
    df_merged.drop(columns=[col + '_new'], inplace=True)  # Eliminar columnas auxiliares

# Guardar el DataFrame actualizado
df_merged.to_csv(output_path, index=False)

print(f"✅ Archivo actualizado guardado en: {output_path}")


✅ Archivo actualizado guardado en: C:\Users\solan\Downloads\get_data_from_songs\data\final_url6.csv


In [43]:
df = pd.read_csv(output_path, low_memory=False)

In [44]:
print("Valores nulos por columna:")
print(df.isnull().sum())


Valores nulos por columna:
artist_name            0
song_name              2
recording_id           0
danceable              0
not_danceable          0
                   ...  
track_uri          63072
playlist_ids       63084
positions          63084
playlists_names    63084
combined_genres     1174
Length: 87, dtype: int64


In [45]:
null_values = df.isnull().sum()
null_values = null_values[null_values > 0]  # Filtrar solo las columnas con nulos
print(null_values.sort_values(ascending=False))  # Ordenar de mayor a menor


playlist_ids          63084
positions             63084
playlists_names       63084
track_uri             63072
views                 43204
language              29082
lyrics                 5235
duration_ms            4451
combined_genres        1174
album_name              480
album_release_date      462
popularity              462
spotify_url             461
song_name                 2
dtype: int64


In [14]:
print("\nNúmero total de filas duplicadas:", df.duplicated().sum())



Número total de filas duplicadas: 0


In [10]:
import pandas as pd

file_total = r'C:\Users\solan\MoodTune\data\procesando\df_80-200_p3_last.csv'
to_add_where_url_null = r'C:\Users\solan\MoodTune\data\procesando\df_80-200_p3_2_spoty.csv'
output_path = r'C:\Users\solan\MoodTune\data\procesando\df_80-200_p3_last_final.csv'

# Columnas que queremos actualizar
columns_to_update = ['album_name', 'popularity', 'album_release_date', 'spotify_url', 'duration_ms']

# Cargar los archivos
df_total = pd.read_csv(file_total, low_memory=False)
df_to_add = pd.read_csv(to_add_where_url_null)

# Fusionar los datasets por `recording_id`
df_merged = df_total.merge(df_to_add[['recording_id'] + columns_to_update], 
                           on='recording_id', how='left', suffixes=('', '_new'))
# Reemplazar solo los valores nulos en `df_total`
for col in columns_to_update:
    df_merged[col] = df_merged[col].combine_first(df_merged[col + '_new'])
    df_merged.drop(columns=[col + '_new'], inplace=True)  # Eliminar columnas auxiliares

# Guardar el DataFrame actualizado
df_merged.to_csv(output_path, index=False)

# Verificar cuántos valores nulos hay en las columnas antes de la actualización
print("Valores nulos antes de la actualización:")
print(df_total[columns_to_update].isnull().sum())

# Verificar cuántos valores nulos quedan después de la actualización
print("Valores nulos después de la actualización:")
print(df_merged[columns_to_update].isnull().sum())


Valores nulos antes de la actualización:
album_name            26751
popularity            26750
album_release_date    26750
spotify_url           26750
duration_ms           26750
dtype: int64
Valores nulos después de la actualización:
album_name            7475
popularity            7472
album_release_date    7472
spotify_url           7472
duration_ms           7472
dtype: int64
